# Train model bàn tay 21 khớp (YOLO11s-Pose) trên Colab Pro

Notebook gọn chỉ để train `hand_pose.pt` cho điều khiển drone bằng tay.

- **Model:** `yolo11s-pose` (lớn hơn bản `n` cũ, chính xác hơn ở tay nhỏ/xa), ảnh 640px.
- **Dữ liệu:** bộ `hand-keypoints` (26.768 ảnh) Ultralytics tự tải. Tùy chọn thêm ảnh tay nhỏ kiểu 10m nếu có `hand-keypoints-10m_aug.zip` trên Drive.
- **Chống mất tiến độ:** mọi kết quả ghi thẳng vào Google Drive. Nếu Colab ngắt giữa chừng, chạy lại từ đầu notebook là tự **train tiếp** từ epoch cuối.

**Trước khi chạy:** Runtime → Change runtime type → GPU (A100 hoặc L4). Sau đó Runtime → Run all.

## 1. Kiểm tra GPU, cài thư viện, gắn Google Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime

import os, glob, shutil, zipfile, yaml
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/Hand_Drone_Models'   # nơi lưu kết quả train
RUN_NAME = 'hand_pose_colab_s640'
PROJECT = os.path.join(DRIVE_DIR, 'runs')
os.makedirs(PROJECT, exist_ok=True)
print('Kết quả sẽ lưu tại:', os.path.join(PROJECT, RUN_NAME))

## 2. Chuẩn bị dữ liệu

- Tải bộ `hand-keypoints` (~370MB) về `/content/datasets`.
- Viết file cấu hình đã sửa:
  - `flip_idx` giữ nguyên thứ tự khớp: lật ngang ảnh bàn tay không đổi ngón cái thành ngón út.
  - `kpt_oks_sigmas` = 0.12 cho cả 21 khớp, giống bản v3 đã train trên máy.
- Nếu Drive có `hand-keypoints-10m_aug.zip` (thư mục `hand-keypoints-10m_aug/images/train` + `labels/train`) thì gộp thêm vào tập train.

In [ ]:
from ultralytics.data.utils import check_det_dataset

info = check_det_dataset('hand-keypoints.yaml')   # tự tải nếu chưa có
root = str(info['path'])
train_dirs = ['images/train']

AUG_ZIP = os.path.join('/content/drive/MyDrive', 'hand-keypoints-10m_aug.zip')
if os.path.exists(AUG_ZIP):
    with zipfile.ZipFile(AUG_ZIP) as z:
        z.extractall(os.path.dirname(root))
    aug_rel = os.path.relpath(os.path.join(os.path.dirname(root), 'hand-keypoints-10m_aug', 'images', 'train'), root)
    train_dirs.append(aug_rel)
    print('Đã thêm ảnh 10m:', len(os.listdir(os.path.join(root, aug_rel))), 'ảnh')
else:
    print('Không có hand-keypoints-10m_aug.zip trên Drive, chỉ dùng hand-keypoints gốc.')

cfg = {
    'path': root,
    'train': train_dirs,
    'val': 'images/val',
    'kpt_shape': [21, 3],
    'flip_idx': list(range(21)),
    'kpt_oks_sigmas': [0.12] * 21,
    'names': {0: 'hand'},
}
DATA_YAML = '/content/hand_keypoints_fixed.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
for d in train_dirs + ['images/val']:
    print(f'{d}: {len(os.listdir(os.path.join(root, d)))} ảnh')

## 3. Train (tự train tiếp nếu đã có `last.pt` trên Drive)

Cấu hình chính:
- `epochs=60`, dừng sớm nếu 15 epoch liền không tốt hơn (`patience=15`).
- `batch=-1`: Ultralytics tự chọn batch vừa ~60% bộ nhớ GPU.
- `scale=0.75`, `mosaic=1.0`: tăng cường xa/gần để bắt được tay nhỏ.
- `close_mosaic=10`: 10 epoch cuối tắt mosaic để khớp tay sắc nét hơn.
- `save_period=1`: lưu checkpoint mỗi epoch lên Drive.

In [ ]:
from ultralytics import YOLO

last_pt = os.path.join(PROJECT, RUN_NAME, 'weights', 'last.pt')
if os.path.exists(last_pt):
    print('Tìm thấy', last_pt, '-> train tiếp')
    model = YOLO(last_pt)
    try:
        model.train(resume=True)
    except AssertionError as e:   # đã train xong hết epoch thì không còn gì để train tiếp
        print('Không train tiếp:', e)
else:
    model = YOLO('yolo11s-pose.pt')
    model.train(
        data=DATA_YAML,
        epochs=60,
        patience=15,
        imgsz=640,
        batch=-1,
        workers=8,
        cos_lr=True,
        mosaic=1.0,
        scale=0.75,
        fliplr=0.5,
        close_mosaic=10,
        project=PROJECT,
        name=RUN_NAME,
        exist_ok=True,
        save_period=1,
        plots=True,
    )

## 4. Kết quả trên tập val

In [ ]:
import pandas as pd
from IPython.display import Image, display

run_dir = os.path.join(PROJECT, RUN_NAME)
df = pd.read_csv(os.path.join(run_dir, 'results.csv'))
df.columns = [c.strip() for c in df.columns]
cols = ['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'metrics/mAP50(P)', 'metrics/mAP50-95(P)']
print(df[cols].tail(5).to_string(index=False))
best = df.loc[df['metrics/mAP50-95(P)'].idxmax()]
print(f"\nEpoch tốt nhất {int(best['epoch'])}: mAP50(P)={best['metrics/mAP50(P)']:.3f}, mAP50-95(P)={best['metrics/mAP50-95(P)']:.3f}")
print('So sánh bản v2 trên máy (yolo11n, 25 epoch): mAP50(P)=0.857, mAP50-95(P)=0.676')

for name in ['results.png', 'val_batch0_pred.jpg']:
    p = os.path.join(run_dir, name)
    if os.path.exists(p):
        display(Image(p, width=900))

## 5. Xuất model và tải về

Lưu `hand_pose.pt` và `hand_pose.onnx` vào `MyDrive/Hand_Drone_Models/` rồi tải về máy.
Trên máy, chép 2 file này đè vào thư mục `Hand-Detection/` (nên giữ lại bản cũ trước khi đè).

In [ ]:
best_pt = os.path.join(PROJECT, RUN_NAME, 'weights', 'best.pt')
out_pt = os.path.join(DRIVE_DIR, 'hand_pose.pt')
shutil.copy(best_pt, out_pt)

onnx_path = YOLO(best_pt).export(format='onnx', imgsz=640, simplify=True)
out_onnx = os.path.join(DRIVE_DIR, 'hand_pose.onnx')
shutil.copy(onnx_path, out_onnx)
print('Đã lưu:', out_pt, out_onnx)

from google.colab import files
files.download(out_pt)
files.download(out_onnx)